# Export MSL Fine-Tune Checkpoints to ExecuTorch (.pte)

This notebook exports the 98-class MSL fine-tuned model to ExecuTorch format for mobile deployment.

## Input
- Checkpoints from: `/content/drive/MyDrive/HazProML/models/msl_finetune_98class/`

## Output
- `yolox_msl_finetune_epoch60.pte` (~19.4 MB)

## What's Special About This Model
- 98 classes (97 original + militaryShippingLabel)
- Fine-tuned from confidence boost checkpoint (100 base + 60 confidence boost + 15 head warmup + 60 full fine-tune)
- 2x objectness loss weight
- MSL detection at ~42% recall, 52% avg confidence, 0 false positives

## Cell 1: Install Dependencies

In [ ]:
!pip install executorch torch torchvision -q
!pip install opencv-python-headless numpy -q

import torch
import numpy as np
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Cell 2: Mount Google Drive & Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# =============================================================================
# CONFIGURATION
# =============================================================================

# Source directory with MSL fine-tuned checkpoints
CHECKPOINT_DIR = '/content/drive/MyDrive/HazProML/models/msl_finetune_98class'

# Checkpoints to export (epochs)
EPOCHS_TO_EXPORT = [60]

# Output directory for ExecuTorch models
OUTPUT_DIR = '/content/drive/MyDrive/HazProML/models/executorch_msl_finetune'

# Model configuration
NUM_CLASSES = 98
INPUT_SIZE = 640
DEPTH = 0.33  # YOLOX-Tiny
WIDTH = 0.375  # YOLOX-Tiny

# =============================================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Epochs to export: {EPOCHS_TO_EXPORT}")
print(f"Num classes: {NUM_CLASSES}")

# Check which checkpoints exist
print("\nAvailable checkpoints:")
for epoch in EPOCHS_TO_EXPORT:
    path = f"{CHECKPOINT_DIR}/msl_finetune_epoch_{epoch}.pth"
    exists = os.path.exists(path)
    size = os.path.getsize(path) / 1024 / 1024 if exists else 0
    status = f"{size:.1f} MB" if exists else "NOT FOUND"
    print(f"  Epoch {epoch}: {status}")

# Also check for final checkpoint
final_path = f"{CHECKPOINT_DIR}/msl_finetune_final.pth"
if os.path.exists(final_path):
    size = os.path.getsize(final_path) / 1024 / 1024
    print(f"  Final: {size:.1f} MB")

# Check class mapping
mapping_path = f"{CHECKPOINT_DIR}/class_mapping_98class.json"
if os.path.exists(mapping_path):
    print(f"  Class mapping: FOUND")
else:
    print(f"  Class mapping: NOT FOUND")

## Cell 3: Setup YOLOX

In [ ]:
import sys

if not os.path.exists('/content/YOLOX'):
    !git clone https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX

!pip install loguru thop ninja tabulate -q

%cd /content/YOLOX
sys.path.insert(0, '/content/YOLOX')

print("YOLOX setup complete!")

## Cell 4: Define Model Architecture

In [ ]:
import torch
import torch.nn as nn
from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead

def create_yolox_tiny(num_classes):
    """Create YOLOX-Tiny model architecture."""
    in_channels = [256, 512, 1024]

    backbone = YOLOPAFPN(
        depth=DEPTH,
        width=WIDTH,
        in_channels=in_channels,
        act='silu'
    )

    head = YOLOXHead(
        num_classes=num_classes,
        width=WIDTH,
        in_channels=in_channels,
        act='silu'
    )

    model = YOLOX(backbone, head)
    return model

class YOLOXExportWrapper(nn.Module):
    """Wrapper for YOLOX model export to ExecuTorch."""
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.model.eval()
        if hasattr(self.model.head, 'decode_in_inference'):
            self.model.head.decode_in_inference = False

    def forward(self, x):
        return self.model(x)

print(f"Model architecture defined: YOLOX-Tiny with {NUM_CLASSES} classes")

## Cell 5: Export Checkpoints

In [ ]:
from torch.export import export
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

exported_files = []

for epoch in EPOCHS_TO_EXPORT:
    print("="*60)
    print(f"EXPORTING MSL FINE-TUNE EPOCH {epoch}")
    print("="*60)

    checkpoint_path = f"{CHECKPOINT_DIR}/msl_finetune_epoch_{epoch}.pth"
    output_filename = f"yolox_msl_finetune_epoch{epoch}.pte"
    output_path = os.path.join(OUTPUT_DIR, output_filename)

    if not os.path.exists(checkpoint_path):
        print(f"  SKIPPED: Checkpoint not found at {checkpoint_path}")
        continue

    try:
        # Create fresh model
        print(f"  Creating model...")
        model = create_yolox_tiny(num_classes=NUM_CLASSES)

        # Load checkpoint
        print(f"  Loading checkpoint...")
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        state_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint
        model.load_state_dict(state_dict, strict=False)
        model.eval()

        # Create export wrapper
        export_model = YOLOXExportWrapper(model)
        export_model.eval()

        # Test forward pass
        dummy_input = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE)
        with torch.no_grad():
            test_output = export_model(dummy_input)
        print(f"  Output shape: {test_output.shape}")
        expected = (1, 8400, 5 + NUM_CLASSES)
        assert test_output.shape == torch.Size(expected), f"Expected {expected}, got {test_output.shape}"
        print(f"  Output verified: [1, 8400, {5 + NUM_CLASSES}]")

        # Export
        print(f"  Exporting with torch.export...")
        example_input = (torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE),)
        exported_program = export(export_model, example_input)

        print(f"  Lowering to edge with XNNPACK...")
        try:
            edge_program = to_edge_transform_and_lower(
                exported_program,
                partitioner=[XnnpackPartitioner()]
            )
        except Exception as e:
            print(f"  XNNPACK failed, using basic edge: {e}")
            from executorch.exir import to_edge
            edge_program = to_edge(exported_program)

        print(f"  Converting to ExecuTorch...")
        executorch_program = edge_program.to_executorch()

        print(f"  Saving .pte file...")
        with open(output_path, 'wb') as f:
            f.write(executorch_program.buffer)

        file_size = os.path.getsize(output_path) / (1024 * 1024)
        print(f"  SUCCESS: {output_filename} ({file_size:.2f} MB)")
        exported_files.append((epoch, output_filename, file_size))

    except Exception as e:
        print(f"  FAILED: {e}")
        import traceback
        traceback.print_exc()

    print()

## Cell 6: Create Metadata & Copy Class Mapping

In [ ]:
import json
import shutil

for epoch, filename, size in exported_files:
    metadata = {
        "model_name": f"yolox_msl_finetune_epoch{epoch}",
        "model_type": "YOLOX-Tiny",
        "training_type": "msl_finetune",
        "training_history": {
            "base_epochs": 100,
            "confidence_boost_epochs": 60,
            "msl_phase1_epochs": 15,
            "msl_phase2_epochs": epoch
        },
        "msl_finetune_settings": {
            "obj_loss_weight": 2.0,
            "msl_oversample": 10,
            "msl_datasets": 2,
            "msl_unique_images": 60,
            "msl_class_id": 97,
            "msl_class_name": "militaryShippingLabel"
        },
        "num_classes": NUM_CLASSES,
        "input_size": INPUT_SIZE,
        "input_format": "NCHW",
        "input_channels": "BGR",
        "input_range": "0-255",
        "output_format": "[batch, anchors, 5+classes]",
        "output_shape": [1, 8400, 5 + NUM_CLASSES],
        "anchors": {
            "total": 8400,
            "strides": [8, 16, 32],
            "grid_sizes": [80, 40, 20]
        },
        "preprocessing": {
            "letterbox": True,
            "pad_value": 114,
            "normalize": False
        },
        "checkpoint_source": f"{CHECKPOINT_DIR}/msl_finetune_epoch_{epoch}.pth",
        "export_format": "ExecuTorch",
        "backend": "XNNPACK",
        "file_size_mb": round(size, 2)
    }

    metadata_path = os.path.join(OUTPUT_DIR, f"yolox_msl_finetune_epoch{epoch}_metadata.json")
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"Created: {metadata_path}")

# Copy 98-class mapping
mapping_src = f"{CHECKPOINT_DIR}/class_mapping_98class.json"
if os.path.exists(mapping_src):
    shutil.copy(mapping_src, f"{OUTPUT_DIR}/class_mapping_98class.json")
    print(f"Copied class_mapping_98class.json")
else:
    print(f"WARNING: class_mapping_98class.json not found at {mapping_src}")

## Cell 7: Summary & Next Steps

In [ ]:
print("="*60)
print("EXPORT COMPLETE!")
print("="*60)
print(f"\nExported {len(exported_files)} MSL fine-tuned model(s) to:")
print(f"  {OUTPUT_DIR}")
print("\nFiles created:")
for epoch, filename, size in exported_files:
    print(f"  - {filename} ({size:.2f} MB)")

print(f"""
{'=' * 60}
APP INTEGRATION STEPS
{'=' * 60}

1. Download from Google Drive:
   {OUTPUT_DIR}/yolox_msl_finetune_epoch60.pte
   {OUTPUT_DIR}/class_mapping_98class.json

2. Copy .pte to React Native project:
   assets/models/yolox_msl_finetune_epoch60.pte

3. Update class_mapping.json:
   - Add class 97: militaryShippingLabel
   - Or replace with class_mapping_98class.json

4. Update executorchService.ts:
   - MODEL_PATH -> 'yolox_msl_finetune_epoch60.pte'
   - NUM_CLASSES -> {NUM_CLASSES}

5. Rebuild:
   npx expo run:ios
   npx expo run:android
""")

# List all files in output directory
print("All files in output directory:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(fpath) / 1024 / 1024
    print(f"  {f} ({size:.2f} MB)")